In [1]:
import pandas as pd
import numpy as np

# 1. Load the historical safe data
df_safe = pd.read_csv('../data/cleaned_water_data.csv')

# Dynamically extract all 17+ Raw Water (RW) metrics
rw_cols = [col for col in df_safe.columns if col.startswith('RW')]
df_real = df_safe[rw_cols].copy()
df_real['Water_Status'] = 1  # 1 = SAFE / NORMAL

# 2. Define the CPHEEO "Red Line" Limits for Raw Water
# (Adjust these slightly if your specific plant has different intake tolerances)
cpheeo_limits = {
    'RW Tur': 30.0,              # Turbidity max NTU
    'RW Colour': 20.0,           # Colour max Hazen
    'RW TDS': 500.0,             # TDS max mg/l
    'RW Iron': 1.0,              # Iron max mg/l
    'RW  Hardness': 300.0,       # Hardness max mg/l (Note the double space from your data)
    'RW S Solids': 50.0,         # Suspended Solids max
    'RW Aluminium': 0.5,         # Aluminium max
    'RW Chloride': 250.0,        # Chloride max
    'RW Manganese': 0.5,         # Manganese max
    'RW Conductivity': 1000.0,   # Conductivity max µS/cm
    'RW Calcium': 75.0,          # Calcium max
    'RW Magnesium': 30.0,        # Magnesium max
    'RW Alkalinity': 120.0,      # Alkalinity max
    'RW Ammonia as N': 0.5       # Ammonia max
}

np.random.seed(42)

# =====================================================================
# TIER 1: Borderline Single-Metric Failures (Sneaky Toxicity)
# We make 150 days where mostly everything is fine, but ONE random chemical slightly fails.
# =====================================================================
df_borderline = df_real.sample(n=150, replace=True).copy()

for idx in df_borderline.index:
    # Pick one random chemical to fail for this specific day
    target_chem = np.random.choice(list(cpheeo_limits.keys()))
    if target_chem in df_borderline.columns:
        limit = cpheeo_limits[target_chem]
        # Spike it just 5% to 40% over the legal limit!
        df_borderline.at[idx, target_chem] = limit * np.random.uniform(1.05, 1.40)

# Handle pH borderline failures separately (pH must be between 6.5 and 8.5)
df_borderline['RW pH'] = np.where(np.random.rand(150) > 0.5, 
                                  np.random.uniform(6.0, 6.4),   # Slightly acidic
                                  np.random.uniform(8.6, 9.0))   # Slightly basic

# =====================================================================
# TIER 2: Moderate Real-World Disasters (Correlated Spikes)
# We make 100 days simulating specific environmental events.
# =====================================================================
df_moderate = df_real.sample(n=100, replace=True).copy()

# A. Mudslide Event (50 days): Spikes physical dirt
mud_mask = np.random.choice(df_moderate.index, 50, replace=False)
df_moderate.loc[mud_mask, 'RW Tur'] = df_moderate.loc[mud_mask, 'RW Tur'] * np.random.uniform(2.0, 4.0)
df_moderate.loc[mud_mask, 'RW Colour'] = df_moderate.loc[mud_mask, 'RW Colour'] * np.random.uniform(2.0, 4.0)
df_moderate.loc[mud_mask, 'RW S Solids'] = df_moderate.loc[mud_mask, 'RW S Solids'] * np.random.uniform(2.0, 4.0)

# B. Industrial/Chemical Event (50 days): Spikes metals and salts
chem_mask = df_moderate.index.difference(mud_mask)
heavy_metals = ['RW TDS', 'RW Conductivity', 'RW Iron', 'RW Manganese']
for metal in heavy_metals:
    if metal in df_moderate.columns:
        df_moderate.loc[chem_mask, metal] = df_moderate.loc[chem_mask, metal] * np.random.uniform(2.0, 4.0)

# =====================================================================
# TIER 3: Catastrophic Failures (Absolute Nightmares)
# We make 50 days of massive 5x-10x multipliers.
# =====================================================================
df_catastrophe = df_real.sample(n=50, replace=True).copy()
for col in rw_cols:
    if col != 'RW pH': # Don't multiply pH, it works on a 0-14 scale
        df_catastrophe[col] = df_catastrophe[col] * np.random.uniform(5.0, 10.0)

# Extreme pH
df_catastrophe['RW pH'] = np.where(np.random.rand(50) > 0.5, np.random.uniform(2.0, 4.0), np.random.uniform(11.0, 13.0))

# =====================================================================
# THE FINAL MIX
# =====================================================================
# 1. Combine all toxic tiers and label them as 0 (FAIL)
df_toxic = pd.concat([df_borderline, df_moderate, df_catastrophe], axis=0)
df_toxic['Water_Status'] = 0

# 2. Combine with the historical safe data (1)
df_master = pd.concat([df_real, df_toxic], axis=0)

# 3. SHUFFLE THE DECK! (frac=1 randomizes the row order entirely)
df_master = df_master.sample(frac=1, random_state=42).reset_index(drop=True)

# 4. Save to CSV
output_path = '../data/rw_classification_data.csv'
df_master.to_csv(output_path, index=False)


print(f"Total Raw Water Metrics Utilized: {len(rw_cols)}")
print(f"Real Safe Days (Class 1): {len(df_real)}")
print(f" Toxic Days Generated (Class 0): {len(df_toxic)}")
print(f"   - Borderline (Sneaky) Fails: {len(df_borderline)}")
print(f"   - Moderate (Correlated) Fails: {len(df_moderate)}")
print(f"   - Catastrophic Fails: {len(df_catastrophe)}")
print(f"Total Shuffled Rows: {len(df_master)}")
print(f" Data saved to: {output_path}")

Total Raw Water Metrics Utilized: 15
Real Safe Days (Class 1): 422
 Toxic Days Generated (Class 0): 300
   - Borderline (Sneaky) Fails: 150
   - Moderate (Correlated) Fails: 100
   - Catastrophic Fails: 50
Total Shuffled Rows: 722
 Data saved to: ../data/rw_classification_data.csv


C:\Users\Vijayasimha\AppData\Local\Temp\ipykernel_29612\661758321.py:45: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '134.47220649808662' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df_borderline.at[idx, target_chem] = limit * np.random.uniform(1.05, 1.40)
C:\Users\Vijayasimha\AppData\Local\Temp\ipykernel_29612\661758321.py:45: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '84.08035714928769' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df_borderline.at[idx, target_chem] = limit * np.random.uniform(1.05, 1.40)
C:\Users\Vijayasimha\AppData\Local\Temp\ipykernel_29612\661758321.py:45: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '41.26182534959702' has dtyp